In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Thu May 14 07:43:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip uninstall -y transformers trl peft bitsandbytes accelerate triton

!pip install -q \
    "torch==2.3.0" \
    "transformers==4.41.0" \
    "peft==0.11.1" \
    "accelerate==0.30.1" \
    "trl==0.8.6" \
    "datasets==2.19.1" \
    "sentencepiece" \
    "bitsandbytes==0.43.1" \
    "triton==2.3.0" \
    "fsspec==2024.3.1"

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0

In [ ]:
import torch
import bitsandbytes as bnb
import triton

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BitsAndBytes:", bnb.__version__)
print("Triton:", triton.__version__)

Torch: 2.3.0+cu121
CUDA available: True
GPU: Tesla T4
BitsAndBytes: 0.43.1
Triton: 2.3.0


In [ ]:
from huggingface_hub import login
login(token="hf_xxxxxxxxx")

In [ ]:
from google.colab import files
import os

print("File picker will open — select fine_tune_dataset.jsonl")
uploaded = files.upload()

print("Size  :", os.path.getsize("fine_tune_dataset.jsonl"), "bytes")
print("Records:", sum(1 for _ in open("fine_tune_dataset.jsonl")))

File picker will open — select fine_tune_dataset.jsonl


Saving fine_tune_dataset.jsonl to fine_tune_dataset.jsonl
Size  : 2912317 bytes
Records: 301


In [ ]:
import json
import random

from datasets import Dataset

with open("fine_tune_dataset.jsonl") as f:
    raw = [json.loads(line) for line in f]

random.seed(42)
random.shuffle(raw)

split = int(len(raw) * 0.9)

train_data = raw[:split]
test_data = raw[split:]

print(f"Train: {len(train_data)} | Test: {len(test_data)}")

Train: 270 | Test: 31


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

GPU: Tesla T4
VRAM: 15.6 GB
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


In [ ]:
MAX_INPUT_CHARS = 2500
MAX_OUTPUT_CHARS = 2500

def truncate_text(text, max_chars):
    if len(text) > max_chars:
        return text[:max_chars]
    return text


def format_prompt(record):

    instruction = (
        record['instruction'] +
        "\nPreserve the original functionality and structure as closely as possible."
    )

    input_code = truncate_text(
        record["input"],
        MAX_INPUT_CHARS
    )

    output_code = truncate_text(
        record["output"],
        MAX_OUTPUT_CHARS
    )

    return (
        f"### Instruction:\n"
        f"{instruction}\n\n"
        f"### Input:\n```\n"
        f"{input_code}\n```\n\n"
        f"### Response:\n```\n"
        f"{output_code}\n```"
        f"{tokenizer.eos_token}"
    )

train_dataset = Dataset.from_list([
    {"text": format_prompt(r)}
    for r in train_data
])

test_dataset = Dataset.from_list([
    {"text": format_prompt(r)}
    for r in test_data
])

print("\n--- Sample Prompt Preview ---\n")
print(train_dataset[0]["text"][:1000])

lengths = [len(x["text"]) for x in train_dataset]

print("\nMax chars:", max(lengths))
print("Avg chars:", sum(lengths)/len(lengths))


--- Sample Prompt Preview ---

### Instruction:
You are a security code reviewer. The following JavaScript (Express.js) code contains these vulnerabilities:
CWE-798: Use of Hard-coded Credentials [Severity: MEDIUM]
CWE-352: Cross-Site Request Forgery (CSRF) [Severity: LOW]
CWE-798: Use of Hard-coded Credentials [Severity: HIGH]
CWE-390: Detection of Error Condition Without Action (Empty Catch) [Severity: LOW]
CWE-347: Improper Verification of Cryptographic Signature (JWT) [Severity: HIGH]
Rewrite the code to fix ALL listed vulnerabilities. Return ONLY the fixed code with no explanation.
Preserve the original functionality and structure as closely as possible.

### Input:
```
const express = require('express');
const multer = require('multer');
const path = require('path');

const app = express();

// Configure storage
const storage = multer.diskStorage({
  destination: function (req, file, cb) {
    cb(null, 'uploads/'); // directory where files will be saved
  },
  filename: function

In [ ]:
lengths = [len(x["text"]) for x in train_dataset]

print("Max chars:", max(lengths))
print("Avg chars:", sum(lengths) / len(lengths))

Max chars: 6102
Avg chars: 4783.3074074074075


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Save directly to Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/security-enhancer-lora-v2"

# Response-only loss
collator = DataCollatorForCompletionOnlyLM(
    response_template="### Response:",
    tokenizer=tokenizer,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # =========================
    # Training
    # =========================
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    # =========================
    # Optimization
    # =========================
    learning_rate=1e-4,
    warmup_steps=50,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",

    # =========================
    # Precision
    # =========================
    fp16=True,

    # =========================
    # Stability
    # =========================
    max_grad_norm=0.3,
    group_by_length=True,

    # =========================
    # Logging / Evaluation
    # =========================
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=25,

    # =========================
    # Saving
    # =========================
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    # =========================
    # Misc
    # =========================
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    dataset_text_field="text",

    # IMPORTANT CHANGE
    max_seq_length=1536,

    data_collator=collator,
    args=training_args,
    peft_config=lora_config,
)

print("=" * 60)
print("Training Configuration")
print("=" * 60)

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples : {len(test_dataset)}")
print(f"Output dir   : {OUTPUT_DIR}")

print("\nTraining started...")
print("Expected:")
print("- Initial loss: ~2-4")
print("- Final loss: ~0.7-1.5")
print("- Runtime: ~2.5-4 hours on T4")

trainer.train()

print("\nSaving model...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\n✅ Training complete")
print(f"Saved to: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


Training Configuration
Train samples: 270
Eval samples : 31
Output dir   : /content/drive/MyDrive/security-enhancer-lora-v2

Training started...
Expected:
- Initial loss: ~2-4
- Final loss: ~0.7-1.5
- Runtime: ~2.5-4 hours on T4


/usr/local/lib/python3.12/dist-packages/trl/trainer/utils.py:141: UserWarning: Could not find response key `### Response:` in the following instance: <s> ### Instruction:
You are a security code reviewer. The following Python (Flask) code contains these vulnerabilities:
CWE-94: Code Injection / Debug Exposure [Severity: HIGH]
CWE-489: Leftover Debug Code [Severity: HIGH]
CWE-347: Improper Verification of Cryptographic Signature (JWT) [Severity: HIGH]
CWE-605: Multiple Binds to the Same Port (Bind-All Interfaces) [Severity: HIGH]
CWE-521: Weak Password Requirements [Severity: MEDIUM]
CWE-668: Exposure of Resource to Wrong Sphere [Severity: MEDIUM]
CWE-798: Use of Hard-coded Credentials [Severity: HIGH]
CWE-390: Detection of Error Condition Without Action (Empty Catch) [Severity: LOW]
CWE-489: Leftover Debug Code [Severity: MEDIUM]
CWE-798: Use of Hard-coded Credentials [Severity: MEDIUM]
CWE-605: Multiple Binds to the Same Port (Bind-All Interfaces) [Severity: MEDIUM]
Rewrite the code t

Step,Training Loss,Validation Loss
25,0.000000,nan
50,0.000000,nan
75,0.000000,nan


Streaming output truncated to the last 5000 lines.
            "email": user.email
        }, 200
    
    return {"error": "User not found"}, 404

# Example Route Usage
@app.route('/user/<int:user_id>', methods=['GET'])
def user_detail(user_id):
    data, status_code = get_user_by_id(user_id)
    return jsonify(data), status_code
```

### Response:
```
from flask import abort, g
from models import db, User

def get_user_profile(user_id_input):
    """
    Fetches a user record securely.
    Ensures input validation, authorization, and data minimization.
    """
    
    # 1. Validate Input: Must be a positive integer
    try:
        user_id = int(user_id_input)
        if user_id <= 0:
            raise ValueError
    except (TypeError, ValueError):
        # We return a 400 without exposing internal logic
        abort(400, description="Invalid user ID format.")

    # 2. Authorization Check: Self-access or Admin only
    # Assumes g.current_user is populated by your auth middleware